In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
import csv

# -----------------------------------------
# Paths
# -----------------------------------------
input_folder = "<PRIVATE_DATA_PATH>"
output_csv = "<PRIVATE_DATA_PATH>"

csv_files = [f for f in os.listdir(input_folder) if f.endswith(".csv")]
print(f"Found {len(csv_files)} metadata CSVs.")


# -----------------------------------------
# Safe extraction of Z from ImagePositionPatient
# -----------------------------------------
def safe_extract_z(value):
    try:
        # Expected format: "[x, y, z]"
        value = value.strip()
        if value.startswith("[") and value.endswith("]"):
            nums = value[1:-1].split(",")
            return float(nums[2])
    except Exception:
        return None  # malformed value
    return None


# -----------------------------------------
# Worker function to process ONE CSV
# -----------------------------------------
def process_csv(csv_name):
    path = os.path.join(input_folder, csv_name)

    try:
        df = pd.read_csv(
            path,
            dtype=str,
            keep_default_na=True,
            low_memory=False
        )

        # Convert Z
        df["Z"] = df["ImagePositionPatient"].apply(safe_extract_z)

        # Build output rows (vectorized, no slow loops)
        result = pd.DataFrame({
    "csv_file": csv_name,
    "pat_id": df["pat_id"],
    "tar_name": df["tar_name"],
    "save_name": df["save_name"],
    "row_num": df["row_num"],

    # anatomical metadata
    "BodyPartExamined": df.get("BodyPartExamined", pd.NA),
    "SeriesDescription": df.get("SeriesDescription", pd.NA),
    "StudyDescription": df.get("StudyDescription", pd.NA),
    "ProtocolName": df.get("ProtocolName", pd.NA),
    "PerformedProtocolCodeSequence": df.get("PerformedProtocolCodeSequence", pd.NA),
    "ScheduledProtocolCodeSequence": df.get("ScheduledProtocolCodeSequence", pd.NA),

    # geometry / position
    "Z": df["Z"],
    "ImageOrientationPatient": df.get("ImageOrientationPatient", pd.NA),
    "PixelSpacing": df.get("PixelSpacing", pd.NA),
    "SliceThickness": df.get("SliceThickness", pd.NA),

    # image type classification
    "ImageType": df.get("ImageType", pd.NA),

    # acquisition details
    "SeriesNumber": df.get("SeriesNumber", pd.NA),
    "AcquisitionNumber": df.get("AcquisitionNumber", pd.NA),
    "ConvolutionKernel": df.get("ConvolutionKernel", pd.NA),
})

        return result

    except Exception as e:
        print(f"[ERROR] Failed on {csv_name}: {e}")
        return pd.DataFrame()


# -----------------------------------------
# Parallel execution
# -----------------------------------------
if __name__ == "__main__":

    workers = min(cpu_count(), 8)  # GPFS-safe limit
    results = []

    with Pool(processes=workers) as pool:
        for df_part in tqdm(
            pool.imap_unordered(process_csv, csv_files, chunksize=1),
            total=len(csv_files),
            desc="Extracting BPE + Z (parallel)"
        ):
            if df_part is not None and not df_part.empty:
                results.append(df_part)

    # Merge all parts
    if results:
        final_df = pd.concat(results, ignore_index=True)
        final_df.to_csv(output_csv, index=False,quoting=csv.QUOTE_ALL)
        print(f"✅ Finished! Saved training data → {output_csv}")
        print(final_df.head())
    else:
        print("❌ No data extracted!")


In [ ]:
df = pd.read_csv(output_csv)
df.head()

In [3]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------------
# Canonical label normalization helpers
# -------------------------------------------------------------------

BAD_BPE = {"", "NONE", "UNKNOWN", "N/A"}

ABDPV_SYNONYMS = {
    "ABDOMEN PELVIS",
    "ABDOMEN  PELVIS",
    "ABDOMENPEL",
    "ABDOMEN PEL",
    "ABDOMEN PEL ",
    "ABD PEL",
    "AB PEL",
    "AB PE",
    "ABDOMENN PELVIS",
    "ABDOMENPELVIS",
}

SPINE_SYNONYMS = {
    "SPINE", "TSPINE", "THORACIC SPINE",
    "LSPINE", "CSPINE", "L-SPINE", "T-SPINE", "C-SPINE"
}

EXTREMITY_SYNONYMS = {
    "EXTREMITY", "UPPER EXTREMITY", "LOWER EXTREMITY",
    "ARM", "LEG"
}


def normalize_bpe(bpe_raw):
    """
    Normalize BodyPartExamined into a small set of canonical labels.
    Returns one of:
        'ABDOMEN', 'ABDOMEN+PELVIS', 'PANCREAS', 'CHEST', 'HEAD',
        'HEART', 'AORTA', 'SPINE', 'NECK', 'EXTREMITY', 'CHABPE',
        'PELVIS', or None (if unknown / unusable).
    """
    if bpe_raw is None or pd.isna(bpe_raw):
        return None

    bpe = str(bpe_raw).strip().upper()
    if bpe in BAD_BPE:
        return None

    # Explicit mappings first
    if bpe in ABDPV_SYNONYMS:
        return "ABDOMEN+PELVIS"
    if bpe == "ABDOMEN":
        return "ABDOMEN"
    if bpe in {"PELVIS", "PELVI"}:
        return "PELVIS"
    if bpe in {"PANCREAS", "PANCREAS HEAD", "PANCREAS TAIL", "PANCREAS BODY"}:
        return "PANCREAS"
    if bpe in {"CHEST", "THORAX"}:
        return "CHEST"
    if bpe in {"HEAD", "BRAIN"}:
        return "HEAD"
    if bpe in {"HEART", "CARDIAC"}:
        return "HEART"
    if bpe in SPINE_SYNONYMS:
        return "SPINE"
    if bpe in EXTREMITY_SYNONYMS:
        return "EXTREMITY"
    if bpe in {"NECK", "CERVICAL"}:
        return "NECK"
    if "AORTA" in bpe:
        return "AORTA"

    # CHABPE / CAP-like encodings
    if "CHABPE" in bpe or "CHEST ABDOMEN PELVIS" in bpe or "CHEST/ABDOMEN/PELVIS" in bpe:
        return "CHABPE"
    if bpe == "CAP":
        return "CHABPE"
    if bpe == "CHESTABPEL":
        return "CHABPE"
    if bpe == "CHEST_ABDOMEN":
        return "CHABPE"
    if bpe == "ABDOMEN LS SPINE":
        return "SPINE"

    # ABDP shorthand (often Abdomen+Pelvis)
    if "ABDP" in bpe:
        return "ABDOMEN+PELVIS"

    # Substring rules (fallback, still conservative)
    if "PANCREAS" in bpe or "PANCREATIC" in bpe:
        return "PANCREAS"
    if "CHEST" in bpe:
        return "CHEST"
    if "ABDOMEN" in bpe and "PELVIS" in bpe:
        return "ABDOMEN+PELVIS"
    if "ABDOMEN" in bpe:
        return "ABDOMEN"
    if "PELVIS" in bpe:
        return "PELVIS"

    # Unknown or not of interest → None (let text or NaN handle it)
    return None


# -------------------------------------------------------------------
# Main heuristic
# -------------------------------------------------------------------

def infer_body_region(row):
    """
    Heuristic body-region label using:
      1) ImageType (LOCALIZER / REFORMATTED → dropped early)
      2) BodyPartExamined (normalized)
      3) SeriesDescription / StudyDescription / ProtocolName patterns

    Returns one of:
        'ABDOMEN', 'ABDOMEN+PELVIS', 'PANCREAS', 'CHABPE', 'CHEST',
        'HEAD', 'HEART', 'AORTA', 'SPINE', 'NECK', 'EXTREMITY',
        'PELVIS', 'LOCALIZER', 'REFORMATTED', or np.nan.
    """

    # 1) ImageType: early exit for localizers / reformats
    img_type = str(row.get("ImageType", "") or "").upper()
    if "LOCALIZER" in img_type or "SCOUT" in img_type:
        return "LOCALIZER"
    if "MPR" in img_type or "MIP" in img_type or "VR" in img_type:
        return "REFORMATTED"

    # 2) Use (normalized) BodyPartExamined if present
    bpe_norm = normalize_bpe(row.get("BodyPartExamined", None))
    if bpe_norm is not None:
        return bpe_norm

    # 3) Use text fields: SeriesDescription / StudyDescription / ProtocolName
    sd = str(row.get("SeriesDescription", "") or "").upper()
    studyd = str(row.get("StudyDescription", "") or "").upper()
    pn = str(row.get("ProtocolName", "") or "").upper()
    text = " ".join([sd, studyd, pn])

    # ---- PANCREAS / PANCREATITIS / PANC patterns ----
    if (
        "PANCREAS" in text
        or "PANCREATITIS" in text
        or "PANCREATIC" in text
        or "PANC_MASS" in text
        or "DE_PANC" in text
        or "BODY_PANCREATIC" in text
    ):
        return "PANCREAS"

    # ---- CHABPE / CAP (Chest-Abdomen-Pelvis) ----
    if (
        "CHABPE" in text
        or "CHEST ABDOMEN PELVIS" in text
        or "CHEST/ABDOMEN/PELVIS" in text
    ):
        return "CHABPE"

    if "CAP" in text and ("CT" in text or "MULTI" in text):
        return "CHABPE"

    # ---- AORTA ----
    if "AORTA" in text:
        return "AORTA"

    # ---- AAA (abdominal aortic aneurysm) → treat as abdomen ----
    if "AAA" in text:
        return "ABDOMEN"

    # ---- HEART / CARDIAC ----
    if "CARDIAC" in text or "HEART" in text:
        return "HEART"

    # ---- Abdomen + Pelvis patterns ----
    if (
        "A/P" in text
        or " A/P " in text
        or "CT A/P" in text
        or "ABD_PEL" in text
        or "ABD PEL" in text
        or "AB PEL" in text
        or "AB PE" in text
        or "DE_ABD/PEL" in text
        or "ABD/PEL" in text
        or "ABD_PEL_ROUTINE_CONTRAST" in text
        or "ABDOMEN PELVIS" in text
        or "ABDOMENPEL" in text
        or "ABDP" in text    # e.g. "ABDP;W N.I./3D"
    ):
        return "ABDOMEN+PELVIS"

    # ---- Head patterns ----
    if "HEAD" in text or "BRAIN" in text:
        return "HEAD"

    # ---- Chest / Thorax / Lung ----
    if "CHEST" in text or "THORAX" in text or "LUNG" in text:
        # Try to detect combos, otherwise chest only
        if "ABDOMEN" in text or " A/P" in text or "ABDOMEN PELVIS" in text or "ABDP" in text:
            return "CHABPE"
        return "CHEST"

    # ---- Spine ----
    if "SPINE" in text or "TSPINE" in text or "LSPINE" in text or "CSPINE" in text:
        return "SPINE"

    # ---- Neck ----
    if "NECK" in text or "CERVICAL" in text:
        return "NECK"

    # ---- Extremity ----
    if "EXTREMITY" in text or "UPPER EXTREMITY" in text or "LOWER EXTREMITY" in text:
        return "EXTREMITY"

    # ---- Generic ABDOMEN detection (last, conservative) ----
    if (
        "ABDOMEN" in text
        or " ABD " in text
        or text.startswith("ABD ")
        or " ABDOM " in text
    ):
        if "PELVIS" in text or "PELVI" in text:
            return "ABDOMEN+PELVIS"
        return "ABDOMEN"

    # ---- Pelvis-only patterns ----
    if "PELVIS" in text or "PELVI" in text:
        return "PELVIS"

    # 4) Unknown → NaN (safe to drop later)
    return np.nan






final_df["body_region"] = final_df.apply(infer_body_region, axis=1)


In [4]:
print(final_df["body_region"].value_counts(dropna=False))

body_region
ABDOMEN           3436564
REFORMATTED       1188239
CHEST              384479
ABDOMEN+PELVIS      60711
PANCREAS            52401
CHABPE              29416
HEART               19852
LOCALIZER           10975
HEAD                 8765
EXTREMITY            5317
AORTA                5061
SPINE                4021
NECK                 3922
PELVIS               2313
Name: count, dtype: int64


In [5]:
keep_regions = {"PANCREAS", "ABDOMEN", "ABDOMEN+PELVIS", "CHABPE", "AORTA"}
df_keep = final_df[final_df["body_region"].isin(keep_regions)]

In [ ]:
df_keep["pat_id"].drop_duplicates()

In [7]:
df_keep["body_region"].value_counts(dropna=False)

body_region
ABDOMEN           3436564
ABDOMEN+PELVIS      60711
PANCREAS            52401
CHABPE              29416
AORTA                5061
Name: count, dtype: int64

In [8]:
x = df_keep["pat_id"].drop_duplicates()

In [9]:
x.to_csv("list.csv",index=False,quoting=csv.QUOTE_ALL)

In [ ]:
import os
import numpy as np
import pandas as pd
from multiprocessing import Pool, cpu_count
from functools import partial
from tqdm import tqdm
import csv

def process_one_csv(fname, input_folder, output_folder):
    in_path = os.path.join(input_folder, fname)
    out_path = os.path.join(output_folder, fname)

    # Skip if already processed
    if os.path.exists(out_path):
        return fname

    df = pd.read_csv(
            in_path,
            dtype=str,
            keep_default_na=True,
            low_memory=False
        )

    # Add body_region column
    df["body_region"] = df.apply(infer_body_region, axis=1)

    # Save
    df.to_csv(out_path, index=False,quoting=csv.QUOTE_ALL)
    return fname


def main():
    input_folder = "<PRIVATE_DATA_PATH>"
    output_folder = "<PRIVATE_DATA_PATH>"

    os.makedirs(output_folder, exist_ok=True)

    csv_files = [f for f in os.listdir(input_folder) if f.endswith(".csv")]
    csv_files.sort()

    n_workers = min(8, cpu_count())  # adjust 8 → more/less depending on your allocation

    print(f"Found {len(csv_files)} CSV files.")
    print(f"Using {n_workers} workers.")

    worker = partial(process_one_csv, input_folder=input_folder, output_folder=output_folder)

    with Pool(processes=n_workers) as pool:
        for _ in tqdm(pool.imap_unordered(worker, csv_files), total=len(csv_files)):
            pass

    print("✅ Done: all CSVs processed with body_region column.")


if __name__ == "__main__":
    main()


Found 308 CSV files.
Using 8 workers.


100%|██████████| 308/308 [16:02<00:00,  3.13s/it]

✅ Done: all CSVs processed with body_region column.


In [ ]:
from tqdm import tqdm
import os
import pandas as pd
from multiprocessing import Pool, cpu_count
from collections import Counter

# ====================================================================
# Paths
# ====================================================================
csv_folder = r"<PRIVATE_DATA_PATH>"
output_stats = "<PRIVATE_DATA_PATH>"

csv_files = [f for f in os.listdir(csv_folder) if f.endswith(".csv")]
csv_files.sort()

print(f"Found {len(csv_files)} CSV files in {csv_folder}")


# ====================================================================
# Single-file worker
# ====================================================================
def process_one_csv(file_name):
    """
    Read one CSV and return a dict: {body_region: count, ...}
    """
    try:
        file_path = os.path.join(csv_folder, file_name)

        df = pd.read_csv(
            file_path,
            dtype=str,
            keep_default_na=True,
            low_memory=False,
        )

        if "body_region" not in df.columns:
            print(f"[WARN] No body_region column in {file_name}")
            return {}

        # value_counts on this file
        vc = df["body_region"].value_counts(dropna=False)

        # convert to plain dict
        counts_dict = vc.to_dict()
        return counts_dict

    except Exception as e:
        print(f"[ERROR] {file_name}: {e}")
        return {}


# ====================================================================
# Main parallel aggregation
# ====================================================================
if __name__ == "__main__":
    num_workers = min(8, cpu_count())   # adjust if you want more/less
    print(f"Using {num_workers} workers")

    global_counter = Counter()

    with Pool(processes=num_workers) as pool:
        for counts_dict in tqdm(
            pool.imap_unordered(process_one_csv, csv_files, chunksize=1),
            total=len(csv_files),
            desc="Aggregating body_region across CSVs"
        ):
            if counts_dict:
                global_counter.update(counts_dict)

    # Convert to DataFrame
    if global_counter:
        data = []
        for region, n in global_counter.items():
            # normalize None / empty keys if they exist
            if region is None or region == "":
                region_label = "NaN_or_empty"
            else:
                region_label = region

            data.append({"body_region": region_label, "count": int(n)})

        stats_df = pd.DataFrame(data)
        stats_df = stats_df.sort_values("count", ascending=False)

        # Print to screen
        print("\n=== Global body_region distribution (all files) ===")
        print(stats_df.to_string(index=False))

        # Save to CSV
        stats_df.to_csv(output_stats, index=False)
        print(f"\n📊 Saved body_region distribution → {output_stats}")

    else:
        print("No body_region stats collected.")
